# ByteBite - on-device export (v4 EfficientNetB3 -> LiteRT / TFLite)

Converts the trained v4 model into a file the Android app loads from `assets/` and
runs with **no network call**: the phone does the inference, offline, on the device.

Nothing is retrained here. This notebook only **converts, measures, and gates** -
then writes the chosen artifact plus everything the Kotlin side needs to reproduce
the training-time preprocessing exactly.

The conversion itself is three lines. The work is in the three things that silently
break a regression model on the way to a phone:

1. **Input scale.** `keras.applications.EfficientNetB3` carries `Rescaling` and
   `Normalization` *inside* the graph, and `efficientnet.preprocess_input` is a
   no-op. So the exported model wants **raw RGB in 0-255**, not 0-1. Dividing by
   255 in Kotlin is silent and catastrophic - it is the most common bug in this
   port, and it produces plausible-looking numbers rather than an obvious failure.
2. **Target de-standardization.** The v4 head emits five *z-scores*, not grams,
   so the train-only `mu`/`sd` have to travel with the model. The earlier v1 model
   was trained on raw kcal and grams instead. Cell 6 measures which one a head is
   rather than assuming it: reading it the wrong way is off by orders of magnitude.
3. **Geometry.** Training resized the full overhead frame to 300x300 with no crop.
   Centre-cropping on the phone changes the apparent portion size, which is exactly
   what mass and calories are read from.

Each of those is pinned into an exported JSON sidecar and verified by a fixture at
the end, so the Android code cannot drift from the training pipeline without the
check failing.

In [ ]:
# =============================================================================
# CELL 1: CONFIG + THE PRE-REGISTERED SHIPPING RULE
# =============================================================================
%matplotlib inline

import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import json
import time
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

# ------------------------------- Paths --------------------------------------
BASE_DIR     = Path(os.environ.get("NUTRITION5K_DIR", "nutrition5k_dataset"))
METADATA_DIR = BASE_DIR / "metadata"
IMAGERY_DIR  = BASE_DIR / "imagery" / "realsense_overhead"
CAFE1_CSV    = METADATA_DIR / "dish_metadata_cafe1.csv"
CAFE2_CSV    = METADATA_DIR / "dish_metadata_cafe2.csv"

OUTPUT_DIR = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
SPLIT_CSV  = Path(os.environ.get("BYTEBITE_SPLIT_CSV",
                                 OUTPUT_DIR / "data_split_seed42.csv"))  # from the training run
P_V4_MODEL = Path(os.environ.get("BYTEBITE_V4_MODEL", OUTPUT_DIR / "v4_s42.keras"))

EXPORT_DIR = OUTPUT_DIR / "android_export"; EXPORT_DIR.mkdir(exist_ok=True)
ASSETS_DIR = Path(os.environ.get(
    "BYTEBITE_ANDROID_ASSETS", "android/app/src/main/assets"))

# --------------------------- Model contract ---------------------------------
IMG_V4  = (300, 300)   # v4's input; Cell 4 re-reads it from the loaded model

# The split each model line was trained on. The scaler is refit on these exact
# train rows, so exporting a model against the wrong split corrupts every number
# the phone shows. Keyed by the start of the .keras filename.
EXPECTED_SPLIT = {
    "v4_":            {"train": 2281, "val": 488, "test": 490},
    "model1_control": {"train": 2282, "val": 489, "test": 489},
}
# Column order is fixed by the Nutrition5k metadata parser and is baked into the
# Dense(5) head. It is NOT alphabetical. Getting this wrong swaps fat and carbs.
TARGETS = ["calories", "mass", "fat", "carb", "protein"]
UNITS   = ["kcal", "g", "g", "g", "g"]

N_REPRESENTATIVE = 200     # calibration images for int8, TRAIN split only
N_LATENCY        = 30      # host-side timing repeats

# ----------------------- PRE-REGISTERED DECISION RULE -----------------------
# Stated before any number is measured, same convention as the v4 validation gate.
#
# float16 is the default shipping variant: half the size of float32, and the only
# float format the GPU delegate accepts natively.
#
# int8 (4x smaller, the fastest CPU path) replaces it ONLY if post-quantization
# test MAE clears both bars:
#     (a) every nutrient degrades by < 2% relative to float32, AND
#     (b) carbohydrate MAE degrades by < 0.10 g absolute.
#
# (b) is deliberately stricter than (a): carbohydrate drives insulin dosing, so it
# is the one target where a size win is not worth a quality loss. A 4x smaller
# file is not worth a worse carb estimate in a diabetes tool.
GATE_REL_PCT  = 2.0
GATE_CARB_ABS = 0.10

print(f"TensorFlow {tf.__version__} | Keras {keras.__version__}")
print(f"v4 model      : {P_V4_MODEL}")
print(f"android assets: {ASSETS_DIR}")
print("Cell 1 complete.")

In [ ]:
# =============================================================================
# CELL 2: REBUILD TARGETS + SPLIT, REFIT THE TRAIN-ONLY SCALER
# =============================================================================
# The scaler is refit here from the SAME train rows the model saw, so the mu/sd
# shipped to the phone are identical to the ones used at training time. Refitting
# on anything wider than train would leak into the numbers on screen.

META_COLS = ["dish_id", "total_calories", "total_mass",
             "total_fat", "total_carb", "total_protein"]

def load_dish_metadata(csv_path):
    """v1 parser: ragged Nutrition5k CSV -> clean 6-column DataFrame."""
    rows, skipped = [], 0
    with open(csv_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6 or not parts[0].startswith("dish_"):
                skipped += 1
                continue
            try:
                rows.append([parts[0]] + [float(v) for v in parts[1:6]])
            except ValueError:
                skipped += 1
    df = pd.DataFrame(rows, columns=META_COLS)
    print(f"{os.path.basename(csv_path)}: parsed {len(df)} dishes ({skipped} skipped)")
    return df

meta = pd.concat([load_dish_metadata(CAFE1_CSV), load_dish_metadata(CAFE2_CSV)],
                 ignore_index=True)
meta = meta.drop_duplicates(subset="dish_id", keep="first").reset_index(drop=True)
meta = meta[meta["total_calories"] > 0].reset_index(drop=True)

# Reuse the split the training notebook committed. Re-deriving it here would risk
# a different permutation and a quantization set that overlaps test.
assert SPLIT_CSV.exists(), (
    f"{SPLIT_CSV} not found - run the model-comparison notebook first so the "
    f"export calibrates on the same train rows the model was fit on.")
split = pd.read_csv(SPLIT_CSV)

def img_path(dish_id):
    return str(IMAGERY_DIR / dish_id / "rgb.png")

final_df = meta.merge(split, on="dish_id", how="inner").reset_index(drop=True)
final_df["image_path"] = final_df["dish_id"].map(img_path)
final_df = final_df[final_df["image_path"].map(os.path.exists)].reset_index(drop=True)

y_all_raw = final_df[META_COLS[1:]].to_numpy(np.float32)   # cal, mass, fat, carb, protein
idx = {s: np.where(final_df["split"].to_numpy() == s)[0] for s in ("train", "val", "test")}
print("\nsplit sizes: " + " ".join(f"{k}={len(v)}" for k, v in idx.items()))

# ---- hard verification: no dish in two splits -------------------------------
assert final_df["dish_id"].is_unique, "duplicate dish_id after merge"
assert sum(len(v) for v in idx.values()) == len(final_df), "split labels do not cover final_df"

# ---- the scaler must come from the SAME train rows the model was fit on -------
# outputs/data_split_seed42.csv currently holds 2282/489/489, while README.md
# describes the v4 line as 2281/488/490 over 3,259 dishes (one dish dropped for a
# physically implausible label). Those cannot both describe the model being
# exported. It matters here and almost nowhere else: mu/sd are fit on the train
# rows, so a scaler off by even a few dishes shifts every number the phone shows
# by a constant - in real kcal and grams, and invisibly to any on-device test.
expected = next((v for k, v in EXPECTED_SPLIT.items() if P_V4_MODEL.name.startswith(k)), None)
actual = {k: len(v) for k, v in idx.items()}
print()
if expected is None:
    print(f"no recorded split for {P_V4_MODEL.name}; confirm {SPLIT_CSV.name} is the one it was trained on")
elif actual != expected:
    print("!" * 74)
    print(f"WARNING: {SPLIT_CSV.name} does not match the split {P_V4_MODEL.name} was trained on")
    print(f"  loaded:   {actual}")
    print(f"  expected: {expected}")
    print("  mu/sd below are fit on the LOADED split. Point SPLIT_CSV at the split")
    print("  written by that model's training run before exporting.")
    print("!" * 74)
else:
    print(f"split matches the one {P_V4_MODEL.name} was trained on")

class TargetScaler:
    """Z-score using TRAIN-ONLY statistics; inverse before any reported number."""
    def fit(self, y):
        self.mu = y.mean(axis=0)
        sd = y.std(axis=0)
        self.sd = np.where(sd == 0, 1.0, sd)
        return self
    def transform(self, y): return (y - self.mu) / self.sd
    def inverse(self, z):   return z * self.sd + self.mu

scaler = TargetScaler().fit(y_all_raw[idx["train"]])       # TRAIN ONLY - no leakage
rt = scaler.inverse(scaler.transform(y_all_raw[idx["val"]]))
assert np.allclose(rt, y_all_raw[idx["val"]], atol=1e-3), "z-score round-trip failed"

print("\ntrain-only target scaler (this is what ships to the phone):")
for t, u, m, s in zip(TARGETS, UNITS, scaler.mu, scaler.sd):
    print(f"  {t:<8} mu={m:8.2f} {u:<4} sd={s:8.2f} {u}")
print("Cell 2 complete.")

In [ ]:
# =============================================================================
# CELL 3: RECORD THE SOURCE GEOMETRY (so Kotlin can reproduce it)
# =============================================================================
# Training did tf.image.resize(full_frame -> 300x300) with NO crop, which stretches
# whatever aspect ratio the source had. To keep inference in-distribution the phone
# must apply the same stretch to a frame of the same aspect ratio - so the camera
# is configured to the dataset's aspect ratio rather than cropped to square.
# The ratio is measured from the real files rather than assumed.

sample_paths = final_df.loc[idx["train"], "image_path"].head(40).tolist()
shapes = []
for p in sample_paths:
    h, w = tf.io.decode_png(tf.io.read_file(p), channels=3).shape[:2]
    shapes.append((int(w), int(h)))

uniq = sorted(set(shapes))
print("distinct (width, height) over 40 train images:")
for w, h in uniq:
    print(f"  {w} x {h}   aspect {w/h:.4f}   n={shapes.count((w, h))}")

SRC_W, SRC_H = max(set(shapes), key=shapes.count)          # modal geometry
SRC_ASPECT = SRC_W / SRC_H
RESIZE_MODE = "stretch_full_frame"                          # no crop, matches training

if len(uniq) > 1:
    print(f"\nNOTE: mixed source geometry; shipping the modal one ({SRC_W}x{SRC_H}).")
print(f"\nsource {SRC_W}x{SRC_H} (aspect {SRC_ASPECT:.4f}) -> model {IMG_V4[0]}x{IMG_V4[1]}"
      f" via {RESIZE_MODE}")
print("Cell 3 complete.")

In [ ]:
# =============================================================================
# CELL 4: KERAS -> SAVEDMODEL -> THREE TFLITE VARIANTS
# =============================================================================
# Keras 3 no longer converts straight from an in-memory model; model.export()
# writes a SavedModel and the converter reads that. The int8 calibration set is
# drawn from TRAIN ONLY - calibrating on val/test would leak evaluation data into
# the shipped weights.

model = keras.models.load_model(P_V4_MODEL)
print(f"loaded {P_V4_MODEL.name} | params {model.count_params():,}")
IMG_V4 = tuple(int(d) for d in model.input_shape[1:3])       # 300x300 for v4
print(f"model input {IMG_V4[0]}x{IMG_V4[1]}")
assert model.output_shape[-1] == len(TARGETS), f"expected {len(TARGETS)} outputs"

SM_DIR = EXPORT_DIR / "saved_model"
if SM_DIR.exists():
    shutil.rmtree(SM_DIR)
model.export(str(SM_DIR))
print(f"SavedModel -> {SM_DIR}")

def decode_for_model(path):
    """Exactly the eval-time path: decode, resize to 300x300, keep 0-255 floats."""
    img = tf.io.decode_png(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, IMG_V4)
    return tf.cast(img, tf.float32)

rep_paths = (final_df.loc[idx["train"], "image_path"]
             .sample(N_REPRESENTATIVE, random_state=SEED).tolist())

def representative_dataset():
    for p in rep_paths:
        yield [decode_for_model(p)[None, ...].numpy()]

def _convert(configure):
    c = tf.lite.TFLiteConverter.from_saved_model(str(SM_DIR))
    configure(c)
    return c.convert()

def cfg_float32(c):
    pass

def cfg_float16(c):
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.target_spec.supported_types = [tf.float16]

def cfg_int8(c):
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.representative_dataset = representative_dataset
    c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    # uint8 in / float32 out: the model's own input range is already 0-255, so the
    # converter lands on a scale of ~1.0 and a camera byte passes in losslessly.
    # Keeping the OUTPUT float32 matters - quantizing five regression outputs to
    # 256 levels would coarsen the estimate for no size win.
    c.inference_input_type = tf.uint8
    c.inference_output_type = tf.float32

VARIANTS = {"float32": cfg_float32, "float16": cfg_float16, "int8": cfg_int8}

tflite_paths, sizes = {}, {}
for name, cfg in VARIANTS.items():
    t0 = time.time()
    blob = _convert(cfg)
    p = EXPORT_DIR / f"bytebite_v4_{name}.tflite"
    p.write_bytes(blob)
    tflite_paths[name] = p
    sizes[name] = len(blob) / 1e6
    print(f"{name:<8} {sizes[name]:6.2f} MB   (convert {time.time()-t0:5.1f}s)")

print("Cell 4 complete.")

In [ ]:
# =============================================================================
# CELL 5: INTERPRETER HELPERS (+ LiteRT forward-compat)
# =============================================================================
# tf.lite.Interpreter is deprecated in favour of the standalone ai_edge_litert
# package. Prefer it when installed so this notebook keeps running after the
# tf.lite shim is removed; the API is identical.
try:
    from ai_edge_litert.interpreter import Interpreter
    print("using ai_edge_litert.Interpreter")
except ImportError:
    from tensorflow.lite.python.interpreter import Interpreter
    print("using tf.lite Interpreter (ai_edge_litert not installed)")

def open_interp(path, threads=4):
    it = Interpreter(model_path=str(path), num_threads=threads)
    it.allocate_tensors()
    return it, it.get_input_details()[0], it.get_output_details()[0]

def run_batch(path, paths, threads=4):
    """Predict z-scores for a list of image paths, one at a time like the phone does."""
    it, ind, outd = open_interp(path, threads)
    want_uint8 = ind["dtype"] == np.uint8
    out = np.zeros((len(paths), len(TARGETS)), np.float32)
    for i, p in enumerate(paths):
        x = decode_for_model(p)[None, ...].numpy()
        # uint8 input: the converter's scale is ~1.0 / zero_point 0, so rounding the
        # 0-255 float to a byte IS the quantization. Do it explicitly rather than
        # relying on an implicit cast.
        x = np.round(x).astype(np.uint8) if want_uint8 else x.astype(np.float32)
        it.set_tensor(ind["index"], x)
        it.invoke()
        out[i] = it.get_tensor(outd["index"])[0]
    return out

for name, p in tflite_paths.items():
    _, ind, outd = open_interp(p)
    scale, zp = ind["quantization"]
    q = f"scale={scale:.8f} zero_point={zp}" if scale else "none"
    print(f"{name:<8} in {np.dtype(ind['dtype']).name:<7} {tuple(ind['shape'])} "
          f"quant[{q}] -> out {np.dtype(outd['dtype']).name} {tuple(outd['shape'])}")
print("Cell 5 complete.")

In [ ]:
# =============================================================================
# CELL 6: ACCURACY ON THE FULL TEST SET, IN REAL UNITS
# =============================================================================
# Conversion error is reported where it matters - kcal and grams after inverse
# z-scoring - not in standardized units, where 0.13 looks harmless and is not.

test_paths = final_df.loc[idx["test"], "image_path"].tolist()
y_test_raw = y_all_raw[idx["test"]]
print(f"evaluating {len(test_paths)} test dishes per variant\n")

ref_z = model.predict(tf.data.Dataset.from_tensor_slices(test_paths)
                      .map(decode_for_model).batch(16), verbose=0)
# Is the head standardized? Decided from the labels, not from memory: v4 trains on
# z-scores, the v1 model1_control trained on raw units, and the right reading is
# dramatically closer to ground truth than the wrong one.
mae_if_z   = float(np.abs(scaler.inverse(ref_z) - y_test_raw).mean())
mae_if_raw = float(np.abs(ref_z - y_test_raw).mean())
STANDARDIZED = mae_if_z < mae_if_raw
ratio = max(mae_if_z, mae_if_raw) / min(mae_if_z, mae_if_raw)
print(f"head reading: as z-scores MAE {mae_if_z:.2f} | as raw units MAE {mae_if_raw:.2f}"
      f" -> {'standardized' if STANDARDIZED else 'raw units'} ({ratio:.0f}x apart)")
assert ratio > 3, ("neither reading of the head is clearly right - "
                   "check the preprocessing before exporting anything")

def to_real(z):
    return scaler.inverse(z) if STANDARDIZED else z

pred = {"keras": to_real(ref_z)}
for name, p in tflite_paths.items():
    t0 = time.time()
    pred[name] = to_real(run_batch(p, test_paths))
    print(f"  {name} done in {time.time()-t0:.0f}s")

mae = {k: np.abs(v - y_test_raw).mean(axis=0) for k, v in pred.items()}
rows = []
for k in ["keras", "float32", "float16", "int8"]:
    r = {"variant": k, "overall": float(mae[k].mean())}
    r.update({t: float(m) for t, m in zip(TARGETS, mae[k])})
    rows.append(r)
mae_df = pd.DataFrame(rows).set_index("variant")

print("\nTest MAE per nutrient (calories in kcal, rest in g):")
print(mae_df.round(3).to_string())

print("\nDelta vs float32 (positive = worse after quantization):")
delta = (mae_df - mae_df.loc["float32"]).drop(index=["keras", "float32"])
rel = (delta / mae_df.loc["float32"] * 100)
for v in delta.index:
    bits = " ".join(f"{t}={delta.loc[v, t]:+.3f} ({rel.loc[v, t]:+.1f}%)" for t in TARGETS)
    print(f"  {v}: {bits}")

# float32 must be a faithful copy of the Keras model - anything else means the
# conversion itself is broken, not the quantization.
assert np.allclose(mae_df.loc["float32", TARGETS].to_numpy(float),
                   mae_df.loc["keras", TARGETS].to_numpy(float), atol=0.05), \
    "float32 TFLite diverged from Keras - conversion bug, not a quantization effect"
print("\nfloat32 TFLite matches Keras to within 0.05 - conversion is faithful.")
print("Cell 6 complete.")

In [ ]:
# =============================================================================
# CELL 7: SIZE / LATENCY TABLE + THE GATE
# =============================================================================
# Host CPU timings rank the variants; they are NOT phone numbers. A desktop x86
# core with XNNPACK is not a mobile big core, and neither sees the NNAPI/GPU path.
# Real device latency is measured on the phone (see android/README_MODEL.md).

lat_paths = test_paths[:N_LATENCY]
lat = {}
for name, p in tflite_paths.items():
    it, ind, outd = open_interp(p)
    want_uint8 = ind["dtype"] == np.uint8
    xs = []
    for q in lat_paths:
        x = decode_for_model(q)[None, ...].numpy()
        xs.append(np.round(x).astype(np.uint8) if want_uint8 else x)
    it.set_tensor(ind["index"], xs[0]); it.invoke()      # warm up, do not time it
    t0 = time.time()
    for x in xs:
        it.set_tensor(ind["index"], x); it.invoke()
    lat[name] = (time.time() - t0) / len(xs) * 1000

summary = pd.DataFrame({
    "size_MB":         {k: round(sizes[k], 2) for k in tflite_paths},
    "host_ms_per_img": {k: round(lat[k], 1) for k in tflite_paths},
    "overall_MAE":     {k: round(float(mae[k].mean()), 3) for k in tflite_paths},
    "carb_MAE_g":      {k: round(float(mae[k][TARGETS.index("carb")]), 3) for k in tflite_paths},
})
print(summary.to_string())

# ------------------------------- the gate -----------------------------------
rel_worst = float(max(rel.loc["int8", t] for t in TARGETS))
carb_abs  = float(delta.loc["int8", "carb"])
pass_rel, pass_carb = rel_worst < GATE_REL_PCT, carb_abs < GATE_CARB_ABS

print("\nGATE (pre-registered in Cell 1)")
print(f"  (a) worst per-nutrient degradation {rel_worst:+.2f}% "
      f"< {GATE_REL_PCT}%      -> {'PASS' if pass_rel else 'FAIL'}")
print(f"  (b) carbohydrate degradation      {carb_abs:+.3f} g "
      f"< {GATE_CARB_ABS} g -> {'PASS' if pass_carb else 'FAIL'}")

SHIP = "int8" if (pass_rel and pass_carb) else "float16"
print(f"\n  -> SHIPPING VARIANT: {SHIP}  ({sizes[SHIP]:.2f} MB)")
if SHIP == "float16":
    print("     int8 did not clear the gate; float16 ships. The int8 file stays in")
    print("     outputs/android_export/ for the record, not copied into assets/.")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(summary.index, summary["size_MB"], color=["#9AA08E", "#7BA05B", "#2E7BA6"])
ax[0].set_ylabel("APK asset size (MB)"); ax[0].set_title("Model size")
w = 0.16
for j, t in enumerate(TARGETS):
    ax[1].bar(np.arange(len(summary)) + j * w - 2 * w,
              [mae[k][j] / mae["float32"][j] for k in summary.index], w, label=t)
ax[1].axhline(1.0, color="k", lw=0.8)
ax[1].set_xticks(np.arange(len(summary))); ax[1].set_xticklabels(summary.index)
ax[1].set_ylabel("MAE relative to float32"); ax[1].set_title("Accuracy cost")
ax[1].legend(fontsize=7, ncol=5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_android_export_tradeoff.png", dpi=150)
plt.show()
print("Cell 7 complete.")

In [ ]:
# =============================================================================
# CELL 8: WRITE THE ANDROID ASSETS
# =============================================================================
# Three files land in assets/. The model is useless without the other two: the
# sidecar carries the preprocessing contract and the de-standardization stats,
# and the fixture lets the app prove on-device that it reproduced this notebook.

ASSETS_DIR.mkdir(parents=True, exist_ok=True)
for old in ASSETS_DIR.glob("*.tflite"):
    old.unlink()                       # one model per build; the sidecar names it
MODEL_ASSET = f"bytebite_{P_V4_MODEL.stem}.tflite"     # e.g. bytebite_v4_s42.tflite
shutil.copy(tflite_paths[SHIP], ASSETS_DIR / MODEL_ASSET)

sidecar = {
    "model_file": MODEL_ASSET,
    "variant": SHIP,
    "source_keras": P_V4_MODEL.name,
    "exported_by": "notebooks/bytebite_android_export.ipynb",
    "input": {
        "width": IMG_V4[0], "height": IMG_V4[1], "channels": 3,
        "dtype": "uint8" if SHIP == "int8" else "float32",
        # THE important field. EfficientNetB3 normalizes inside the graph.
        "range": "0-255",
        "note": "Do NOT divide by 255 and do NOT apply ImageNet mean/std. The "
                "Rescaling and Normalization layers are inside the exported graph.",
        "channel_order": "RGB",
        "resize_mode": RESIZE_MODE,
        "source_width": SRC_W, "source_height": SRC_H,
        "source_aspect": round(SRC_ASPECT, 6),
        "geometry_note": "Training resized the full frame with no crop. Configure "
                         "the camera to source_aspect and stretch the whole frame; "
                         "a square centre-crop changes apparent portion size.",
    },
    "output": {
        "targets": TARGETS, "units": UNITS,
        "standardized": bool(STANDARDIZED),
        # A raw-unit head ships the identity so the phone's arithmetic is one line.
        "mu": [float(v) for v in scaler.mu] if STANDARDIZED else [0.0] * len(TARGETS),
        "sd": [float(v) for v in scaler.sd] if STANDARDIZED else [1.0] * len(TARGETS),
        "note": "real = z * sd + mu, index-aligned with targets.",
    },
    "test_mae": {t: round(float(mae[SHIP][i]), 4) for i, t in enumerate(TARGETS)},
}
(ASSETS_DIR / "bytebite_model.json").write_text(json.dumps(sidecar, indent=2))

# ---- parity fixture: one real test dish, pinned ----------------------------
fix_i = 0
fix_path = test_paths[fix_i]
fix_img = decode_for_model(fix_path)                       # model-size float 0-255
fix_png = tf.io.encode_png(tf.cast(tf.round(fix_img), tf.uint8)).numpy()
(ASSETS_DIR / "fixture_dish.png").write_bytes(fix_png)

# Expected values are computed from the PNG bytes themselves, not the float resize
# they were rounded from, so the phone and this notebook start from identical
# pixels and any gap is the phone's arithmetic, not the fixture's.
_px = tf.cast(tf.io.decode_png(fix_png, channels=3), tf.float32).numpy()[None, ...]
_it, _ind, _outd = open_interp(tflite_paths[SHIP])
_it.set_tensor(_ind["index"], _px.astype(np.uint8) if _ind["dtype"] == np.uint8 else _px)
_it.invoke()
fix_z = _it.get_tensor(_outd["index"])[0].copy()

fixture = {
    "dish_id": final_df.loc[idx["test"][fix_i], "dish_id"],
    "image_asset": "fixture_dish.png",
    "note": f"Already resized to {IMG_V4[0]}x{IMG_V4[1]}. Decode, read RGB as 0-255, "
            "run, apply real = z * sd + mu, compare to expected_real within tolerance.",
    "expected_z":    [float(v) for v in fix_z],
    "expected_real": [float(v) for v in to_real(fix_z[None, :])[0]],
    "ground_truth":  [float(v) for v in y_test_raw[fix_i]],
    "tolerance_real": [2.0, 2.0, 0.5, 0.5, 0.5],
}
(ASSETS_DIR / "fixture_dish.json").write_text(json.dumps(fixture, indent=2))

for f in sorted(ASSETS_DIR.iterdir()):
    print(f"  {f.name:<24} {f.stat().st_size/1e6:7.3f} MB")
print("Cell 8 complete.")

In [ ]:
# =============================================================================
# CELL 9: END-TO-END VERIFICATION FROM THE ASSETS FOLDER
# =============================================================================
# Reload only what the phone gets - the asset files, nothing from memory - and
# reproduce the fixture through the same arithmetic NutritionEstimator.kt does.
# If this cell passes and the app disagrees, the bug is in the Kotlin.

side = json.loads((ASSETS_DIR / "bytebite_model.json").read_text())
fx   = json.loads((ASSETS_DIR / "fixture_dish.json").read_text())

mu = np.array(side["output"]["mu"], np.float32)
sd = np.array(side["output"]["sd"], np.float32)

png = tf.io.decode_png(tf.io.read_file(str(ASSETS_DIR / fx["image_asset"])), channels=3)
x = tf.cast(png, tf.float32).numpy()[None, ...]            # 0-255, no /255
assert x.shape[1:3] == IMG_V4, f"fixture image is not {IMG_V4[0]}x{IMG_V4[1]}"
assert x.max() > 1.5, "fixture decoded into 0-1 - wrong input range"

it, ind, outd = open_interp(ASSETS_DIR / side["model_file"])
if ind["dtype"] == np.uint8:
    x = np.round(x).astype(np.uint8)
it.set_tensor(ind["index"], x); it.invoke()
z = it.get_tensor(outd["index"])[0]
real = z * sd + mu

print(f"dish {fx['dish_id']}  (variant {side['variant']})")
print(f"{'target':<9}{'predicted':>12}{'expected':>12}{'truth':>10}{'unit':>6}")
for i, t in enumerate(side["output"]["targets"]):
    print(f"{t:<9}{real[i]:>12.2f}{fx['expected_real'][i]:>12.2f}"
          f"{fx['ground_truth'][i]:>10.2f}{side['output']['units'][i]:>6}")

# Per-target tolerances, compared explicitly: numpy's assert_allclose cannot
# format an array-valued atol in recent releases.
gap = np.abs(real - np.array(fx["expected_real"], np.float32))
tol = np.array(fx["tolerance_real"], np.float32)
assert np.all(gap <= tol), f"fixture mismatch: gap {np.round(gap, 4)} vs tolerance {tol}"
print(f"largest gap to the fixture: {gap.max():.5f}")
print("\nPASS - assets reproduce the notebook. This is the contract the Kotlin "
      "fixture test asserts on-device.")
print("Cell 9 complete.")

## What ships, and what is still open

`assets/` ends up with three files, and the app needs all three:

| file | why it exists |
|---|---|
| `bytebite_v4.tflite` | the gated variant - the only file that does arithmetic |
| `bytebite_model.json` | input range, geometry, and the train-only `mu`/`sd` |
| `fixture_dish.png` + `.json` | one pinned dish so the phone can prove it agrees with this notebook |

The sidecar exists so the preprocessing contract lives in **one** place. If a
future run retrains at a different resolution or refits the scaler, the JSON
changes and the Kotlin picks it up with no edit; the fixture fails loudly if the
two ever disagree.

**Not answered here.** Host CPU timings rank the variants but say nothing about a
phone. On-device latency, the NNAPI and GPU delegate comparison, and thermal
behaviour over a run of consecutive scans have to be measured on the device - the
EfficientNetB3 backbone at 300x300 is the heaviest of the three models in the
paper, and the one most likely to need a delegate to stay interactive.

**The depth gap persists on-device.** Nutrition5k ships overhead depth that the v4
model does not use, and mass is the weakest target. A phone with a depth sensor
could close some of that, but it would mean a second input tensor and a retrained
model, not a conversion change.